# Convex hull and collision meshes

Companion to [W10 §4](../README.md#4-convex-hull-the-rubber-band). You will:

1. test which way three points turn, the only geometry the 2D hull needs;
2. build the lower half of a hull point by point;
3. compare with the tested `convex_hull` and with COMPAS;
4. wrap a timber node in a 3D hull, its collision mesh, and see why one hull is not enough.

![A robot's collision meshes: each link wrapped in its convex hull](collision_mesh_spot_anymal.jpeg)

*A legged robot and its collision meshes: each link is checked as its convex hull, not
as its detailed shape.*

**Setup.** Run this cell and move on. It lets the notebook find the drawing helpers
(`viz`) and this week's algorithms (`examples/`), whether it sits in the lecture
folder or is your copy in `MyWork/`.

In [ ]:
import sys
import pathlib

HERE = pathlib.Path.cwd()
SHARED = next(p / "Lecture" / "shared" for p in [HERE, *HERE.parents] if (p / "Lecture" / "shared" / "viz.py").exists())
sys.path[:0] = [str(SHARED), str(SHARED.parent / "Lecture_10" / "examples")]

## Step 1 — Which way do three points turn?

Walk from `o` to `a`, then on to `b`. If you turn left, the cross product
`(a - o) × (b - o)` is positive; turning right, negative; straight on, zero. One
multiplication and one subtraction: this is the whole of the geometry in the 2D hull.

In [ ]:
import convex_hull

o, a = (0, 0), (2, 1)
for b in [(3, 3), (3, 0), (4, 2)]:
    turn = convex_hull.cross(o, a, b)
    print(f"o -> a -> {b}: cross = {turn:>3}   ", "left" if turn > 0 else "right" if turn < 0 else "straight")

## Step 2 — Points to wrap

Thirty points scattered on a site, say the trees a fence must go around.

In [ ]:
import random

from viz import draw_points

rnd = random.Random(7)
points = [(rnd.uniform(0, 10), rnd.uniform(0, 6)) for _ in range(30)]
draw_points(points, "30 points")

## Step 3 — The lower half, point by point

Sort the points left to right. Walk through them, adding each to a chain. Whenever the
last three points of the chain turn right (or go straight), the middle one cannot be on
the hull: drop it, and check again. The snapshots show the chain after 8, 16 and all 30
points; blue is the chain, grey the points not reached yet.

In [ ]:
from viz import row

ordered = sorted(points)
chain, snapshots = [], {}
for k, p in enumerate(ordered, start=1):
    while len(chain) >= 2 and convex_hull.cross(chain[-2], chain[-1], p) <= 0:
        chain.pop()
    chain.append(p)
    if k in (8, 16, 30):
        snapshots[k] = list(chain)

row(*(draw_points(points, f"after {k} points", lines=list(zip(c, c[1:])),
                  colors=["#1c7ed6" if q in c else "#adb5bd" for q in points], size=(320, 240))
      for k, c in snapshots.items()))

## Step 4 — The whole hull

Walking back from right to left gives the upper half the same way; `convex_hull` joins
the two. COMPAS's `convex_hull_xy` must find the same corners.

In [ ]:
from compas.geometry import convex_hull_xy

hull = convex_hull.convex_hull(points)
print(len(hull), "corners; COMPAS finds", len(convex_hull_xy([list(p) + [0] for p in points])))
draw_points(points, "the hull: every point inside the polygon", polygons=[hull])

## Step 5 — A collision mesh in 3D

A robot will place this timber node: three stubs meeting at a corner. Before every
move, the planner checks the node against the frame. `collision_mesh` wraps it in its
3D convex hull (orange). Drag to orbit.

In [ ]:
from compas.colors import Color

from viz import view3d

parts = convex_hull.corner_parts()
node = convex_hull.joined(parts)
one_hull = convex_hull.collision_mesh(node)
view3d((node, {"facecolor": Color.from_hex("#d0ebff")}),
       (one_hull, {"facecolor": Color.from_hex("#ffa94d"), "opacity": 0.35}))

A point is inside a convex mesh exactly when it is behind the plane of every face:
one dot product per face. That is why planners want convex shapes. But compare the
volumes: the single hull fills the empty corner between the stubs.

In [ ]:
solid = sum(part.volume() for part in parts)
print(f"the stubs: {solid:.4f} m³   one hull: {one_hull.volume():.4f} m³   ({one_hull.volume() / solid:.1f} x)")

## Step 6 — One hull per part

The robot's tool will pass through the corner, 0.2 m from each stub. With one hull
the planner thinks it collides; with one hull per stub, it does not. Splitting a shape
into convex parts is what collision tools call convex decomposition.

In [ ]:
part_hulls = [convex_hull.collision_mesh(part) for part in parts]
tool = [0.2, 0.2, 0.2]
print("inside the single hull:   ", convex_hull.inside(one_hull, tool))
print("inside any per-stub hull: ", any(convex_hull.inside(h, tool) for h in part_hulls))
print(f"per-stub hulls: {sum(h.volume() for h in part_hulls):.4f} m³, the stubs exactly")
view3d(*[(h, {"facecolor": Color.from_hex("#8ce99a"), "opacity": 0.6}) for h in part_hulls])

## Your turn

1. In Step 3, change `<= 0` to `< 0`. Add three points on one straight line along the
   bottom. What changes in the chain?
2. Add a fourth stub, a diagonal brace. How much bigger does the single hull get?
3. Move the tool to `[0.05, 0.05, 0.05]`. Which answer changes, and is the robot safe?